In [ ]:
import glob

# Wczytaj wszystkie zapisane wyniki
grid_files = sorted(glob.glob('results/grid_playtime_*.csv'))
purity_files = sorted(glob.glob('results/purity_playtime_*.csv'))
stats_files = sorted(glob.glob('results/stats_playtime_*.csv'))

if len(grid_files) < 2:
    print("Za mało wyników! Uruchom notatnik dla co najmniej 2 różnych progów playtime.")
else:
    # --- Porównanie Modularity ---
    all_grid = pd.concat([pd.read_csv(f) for f in grid_files])
    all_purity = pd.concat([pd.read_csv(f) for f in purity_files])
    all_stats = pd.concat([pd.read_csv(f) for f in stats_files])
    
    fig, axes = plt.subplots(1, 3, figsize=(22, 6))
    
    # 1. Modularity vs Playtime
    all_grid['label'] = all_grid.apply(lambda r: f"{r['Threshold']}|{r['Metric'][:3]}|g={r['Gamma']}", axis=1)
    sns.boxplot(data=all_grid, x='playtime_min', y='modularity', ax=axes[0], palette='Set2')
    axes[0].set_title('Modularity vs Próg Playtime')
    axes[0].set_xlabel('Min. Playtime (minuty)')
    axes[0].set_ylabel('Modularity')
    
    # 2. Purity vs Playtime
    sns.boxplot(data=all_purity, x='playtime_min', y='Weighted_Purity_Top10', ax=axes[1], palette='Set2')
    axes[1].set_title('Genre Purity vs Próg Playtime')
    axes[1].set_xlabel('Min. Playtime (minuty)')
    axes[1].set_ylabel('Weighted Purity')
    
    # 3. Aktywne węzły vs Playtime
    similar_stats = all_stats[all_stats['rel_type'] == 'SIMILAR']
    axes[2].bar(similar_stats['playtime_min'].astype(str), similar_stats['active_nodes'], color='steelblue')
    axes[2].set_title('Aktywne Węzły (SIMILAR) vs Próg Playtime')
    axes[2].set_xlabel('Min. Playtime (minuty)')
    axes[2].set_ylabel('Liczba aktywnych węzłów')
    
    plt.tight_layout()
    plt.show()
    
    print("\nPodsumowanie statystyk sieci dla różnych progów:")
    display(all_stats)
    
    print("\nPodsumowanie Grid Search dla różnych progów:")
    display(all_grid.groupby('playtime_min')[['modularity', 'communityCount']].describe())
